# 02. Zegary i punkty w czasie

## Cele

Po tym notebooku potrafisz:

* odróżnić punkt w czasie (`time_point`) od odcinka czasu (`duration`) i wiedzieć, jakie działania na nich mają sens,
* wybrać zegar: `system_clock` do daty i godziny, `steady_clock` do mierzenia czasu i timeoutów,
* zmierzyć czas wykonania kodu tak, żeby wynik coś znaczył,
* wyjaśnić, dlaczego pętla z `sleep_for` się spóźnia, a z `sleep_until` — nie,
* rozpoznać pułapkę „nieskończonego” timeoutu zapisanego jako `max()`.

## `time_point`: odcinek czasu od epoki

Zegar to trzy rzeczy: typ `duration`, typ `time_point` i funkcja `now()`. Początek odliczania zegara
nazywa się jego **epoką** ([\[time.clock.req\]/1](https://eel.is/c++draft/time.clock.req#1)).
`time_point` przechowuje po prostu odcinek czasu, który upłynął od epoki
([cppreference: `time_point`](https://en.cppreference.com/w/cpp/chrono/time_point.html)) —
ta sama liczba, ale inne znaczenie: `10'000s` to „jakieś 10 000 sekund”, a punkt 10 000 s po epoce
`system_clock` to konkretna chwila: 1970-01-01 02:46:40 UTC.

Arytmetyka przypomina wskaźniki
([Hinnant, s. 80](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf)):

| Działanie | Wynik |
|---|---|
| `time_point - time_point` | `duration` |
| `time_point ± duration` | `time_point` |
| `time_point + time_point` | błąd kompilacji |

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    sys_seconds tp{10'000s};  // sys_seconds = time_point<system_clock, seconds>
    std::cout << "od epoki: " << tp.time_since_epoch().count() << " s\n";

    auto later = tp + 1h;              // punkt + odcinek = punkt
    seconds diff = later - tp;         // punkt - punkt = odcinek
    std::cout << "różnica: " << diff.count() << " s\n";
}
```

Dodawanie dwóch punktów nie ma sensu („wtorek + czwartek”?), więc się nie kompiluje:

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    auto a = steady_clock::now();
    auto b = steady_clock::now();
    auto c = a + b;  // błąd: time_point + time_point
    (void)c;
}
```

Punkty z **różnych zegarów** też się nie mieszają — mają różne epoki, więc porównanie ich liczb byłoby bez sensu:

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    steady_clock::time_point t = system_clock::now();  // błąd: inny zegar
    (void)t;
}
```

## `system_clock` czy `steady_clock`?

Hinnant tłumaczy różnicę obrazowo
([odpowiedź na Stack Overflow](https://stackoverflow.com/a/31553641)): `system_clock` to **zegarek** —
powie, która jest godzina. `steady_clock` to **stoper** — powie, ile trwało okrążenie, ale nie powie, która jest godzina.

| | `system_clock` | `steady_clock` |
|---|---|---|
| Do czego | data i godzina, znaczniki czasu w logach | mierzenie czasu, timeouty, harmonogramy |
| Epoka | 1970-01-01 00:00:00 UTC (czas uniksowy, gwarantowany od C++20) | dowolna, np. start systemu |
| Może się cofnąć? | **tak** — synchronizacja NTP, ręczna zmiana zegara | **nie** — monotoniczny |
| `is_steady` | `false` | `true` |

([cppreference: `system_clock`](https://en.cppreference.com/w/cpp/chrono/system_clock.html#Notes),
[`steady_clock`](https://en.cppreference.com/w/cpp/chrono/steady_clock.html),
[\[time.clock.system.overview\]](https://eel.is/c++draft/time.clock.system.overview#1))

Zmiana czasu letni/zimowy **nie** wpływa na `system_clock` — liczy on czas UTC; strefy czasowe
to osobny temat (notebook 05). Cofnięcie zegara przez NTP — już tak.

A `high_resolution_clock`? Standard pozwala, żeby był po prostu innym zegarem pod inną nazwą
([\[time.clock.hires\]](https://eel.is/c++draft/time.clock.hires#1)) — i tak jest w praktyce:
w libstdc++ (GCC) to `system_clock`, w MSVC — `steady_clock`. Nawet jego autor opowiadał się za jego wycofaniem
([cppreference, Notes](https://en.cppreference.com/w/cpp/chrono/high_resolution_clock.html#Notes)).
Wybieraj wprost `steady_clock` albo `system_clock`, żeby wiedzieć, co dostajesz:

```cpp
#include <chrono>
#include <iostream>
#include <type_traits>

int main() {
    using namespace std::chrono;
    std::cout << "system_clock::is_steady          = " << system_clock::is_steady << '\n';
    std::cout << "steady_clock::is_steady          = " << steady_clock::is_steady << '\n';
    std::cout << "high_resolution_clock::is_steady = " << high_resolution_clock::is_steady << '\n';
    std::cout << "high_resolution_clock == system_clock? "
              << std::is_same_v<high_resolution_clock, system_clock> << '\n';
}
```

(Wynik ostatnich dwóch linii zależy od biblioteki standardowej — i właśnie o to chodzi.)

## Mierzenie czasu

Wzorzec jest prosty: dwa odczyty `steady_clock` i różnica. Wynik od razu zamieniamy na czytelną jednostkę —
`duration<double, std::milli>` przyjmie różnicę niejawnie, bez obcinania:

```cpp
#include <chrono>
#include <iostream>
#include <vector>
#include <numeric>

int main() {
    using Clock = std::chrono::steady_clock;
    std::vector<int> v(1'000'000, 1);

    auto start = Clock::now();
    long long sum = std::accumulate(v.begin(), v.end(), 0LL);
    auto stop = Clock::now();

    std::chrono::duration<double, std::milli> elapsed = stop - start;
    std::cout << "suma " << sum << " w " << elapsed.count() << " ms\n";
}
```

Trudniejsze jest to, **co** mierzymy
([Learn C++: Timing your code](https://www.learncpp.com/cpp-tutorial/timing-your-code/),
[Google Benchmark: Preventing Optimization](https://google.github.io/benchmark/user_guide.html#preventing-optimization)):

* **Build bez optymalizacji** (Debug) mierzy coś innego niż program, który oddasz użytkownikom.
* **Z optymalizacjami** kompilator może usunąć obliczenia, których wyniku nikt nie używa — i zmierzysz pustkę.
* **Pierwsze uruchomienie** bywa wolniejsze (pamięć podręczna, alokacja stron) — powtórz pomiar.
* **Sam zegar** ma koszt i rozdzielczość: dwa kolejne `now()` dzielą dziesiątki–setki nanosekund
  ([Hinnant na Stack Overflow](https://stackoverflow.com/a/42612438)) — bardzo krótkie fragmenty mierz w pętli.

Drugi punkt widać w demonstracji [`demos/bench.cpp`](demos/bench.cpp) (uruchomisz ją niżej).

## `sleep_for` czy `sleep_until`?

Funkcje z końcówką `_for` przyjmują **odcinek** czasu („czekaj 20 ms od teraz”), z końcówką `_until` —
**punkt** („czekaj do 12:00:00.020”). Standard mówi wprost, że każde czekanie trwa
*co najmniej* tyle, ile zamówiono, plus opóźnienie systemu
([\[thread.req.timing\]](https://eel.is/c++draft/thread.req.timing#3),
[cppreference: `sleep_for`](https://en.cppreference.com/w/cpp/thread/sleep_for.html)).

W zadaniu okresowym („co 20 ms”) to opóźnienie — razem z czasem samej pracy — przy `sleep_for`
**dokłada się w każdym obrocie pętli**. Przy `sleep_until` liczymy kolejne terminy od stałego punktu
startu, więc spóźnienie z jednego obrotu nie przechodzi na następny.
Demonstracja [`demos/drift.cpp`](demos/drift.cpp) wykonuje 50 obrotów po 20 ms, każdy z 3 ms „pracy”.

In [ ]:
# Build the demos (Release) and run them: drift (~2 s) and bench
!cmake -S demos -B demos/build && cmake --build demos/build --config Release --target run_demos

Typowy wynik (liczby zależą od maszyny):

```
ideal:       1000 ms
sleep_for:   1170 ms
sleep_until: 1000 ms
result ignored: 0.0001 ms
result used:    250 ms
```

Pętla z `sleep_for` „gubi” prawie 0,2 s na sekundę — 3 ms pracy plus opóźnienie wybudzenia w każdym z 50 obrotów.
A pętla, której wyniku nikt nie użył, „wykonała się” w ułamek mikrosekundy, bo kompilator ją usunął.

## Timeouty i pułapka `max()`

Ta sama para końcówek występuje w całej bibliotece wątków: `try_lock_for` / `try_lock_until`,
`condition_variable::wait_for` / `wait_until`, `future::wait_for` / `wait_until`.
Standard definiuje wersję `_for` przez wersję `_until`
([\[thread.condition.condvar\]/24](https://eel.is/c++draft/thread.condition.condvar#24)):
`wait_for(lock, d)` działa jak `wait_until(lock, steady_clock::now() + d)` — dokładniej `now()` plus `d`
zaokrąglone w górę do tyknięć `steady_clock`
([\[thread.condition.general\]/6](https://eel.is/c++draft/thread.condition.general#6)).

Z tej definicji wynika pułapka. Kusi, żeby „czekaj w nieskończoność” zapisać jako największy możliwy odcinek czasu,
np. `seconds::max()`. Ale `now() + seconds::max()` nie mieści się w liczniku `steady_clock` (nanosekundy, 64 bity) —
to przepełnienie liczby ze znakiem, czyli zachowanie niezdefiniowane. W praktyce wynik „zawija się” w przeszłość,
a czekanie na termin z przeszłości kończy się natychmiast:

```cpp
#include <chrono>
#include <iostream>
#include <thread>

int main() {
    using namespace std::chrono;
    auto start = steady_clock::now();
    std::this_thread::sleep_until(steady_clock::now() + seconds::max());  // "na zawsze"?
    auto waited = duration_cast<milliseconds>(steady_clock::now() - start);
    std::cout << "czekałem " << waited.count() << " ms\n";
}
```

Ten sam błąd zdarzał się w realnych programach
([pytanie na Stack Overflow](https://stackoverflow.com/questions/77794791): serwer zajmował 100% procesora,
bo „wieczne” uśpienie kończyło się od razu), a w bibliotekach standardowych są na niego zgłoszenia
([GCC Bug 113327](https://www.mail-archive.com/gcc-bugs@gcc.gnu.org/msg881194.html),
[GCC Bug 119258](https://www.mail-archive.com/gcc-bugs@gcc.gnu.org/msg854143.html)).
Hinnant radzi pragmatycznie: jeśli potrzebujesz „bardzo długo”, poproś o np. `9000h` (ponad rok) —
to na pewno się nie przepełni ([Stack Overflow](https://stackoverflow.com/a/42639024)).

## Ćwiczenie 2

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [cppreference: `time_point`](https://en.cppreference.com/w/cpp/chrono/time_point.html),
  [`system_clock`](https://en.cppreference.com/w/cpp/chrono/system_clock.html#Notes),
  [`steady_clock`](https://en.cppreference.com/w/cpp/chrono/steady_clock.html),
  [`high_resolution_clock`](https://en.cppreference.com/w/cpp/chrono/high_resolution_clock.html#Notes)
* [cppreference: `sleep_for`](https://en.cppreference.com/w/cpp/thread/sleep_for.html),
  [`sleep_until`](https://en.cppreference.com/w/cpp/thread/sleep_until.html),
  [`condition_variable::wait_for`](https://en.cppreference.com/w/cpp/thread/condition_variable/wait_for.html)
* Standard C++ (szkic roboczy): [\[thread.req.timing\]](https://eel.is/c++draft/thread.req.timing#3) — timeouty względne i bezwzględne
* Howard Hinnant: [różnica między `steady_clock` a `system_clock`](https://stackoverflow.com/a/31553641),
  [który zegar do znaczników czasu w logach](https://stackoverflow.com/a/64084510)
* Andrzej Krzemieński, [„Using std::chrono”](https://akrzemi1.wordpress.com/2022/04/11/using-stdchrono/)
* [N2661 „A Foundation to Sleep On”](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2008/n2661.htm#Goals) —
  pierwotna propozycja `<chrono>` (2008); stąd nazwy `sleep_for` i `sleep_until`

---

[Strona główna](../_start.ipynb)